In [8]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import util

In [9]:
image1 = cv2.imread('../images/webots_lanes/turn-left.png')
image2 = cv2.imread('../images/webots_lanes/straight.png')

h, w, c = image1.shape

In [10]:
def process_image(image, ksize=(3,3), thres1=50, thres2=100, polygon=None):
    img_g = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img_b = cv2.GaussianBlur(img_g, ksize=ksize, sigmaX=0)
    img_e = cv2.Canny(img_b, thres1, thres2)
    
    if polygon is not None:
        roi = np.array([polygon], dtype=np.int32)
        mask = np.zeros_like(img_e)
        cv2.fillPoly(mask, pts=roi, color=(255,255,255))
        img_k = cv2.bitwise_and(img_e, mask)
        return img_g, img_b, img_e, img_k
    
    return img_g, img_b, img_e

In [11]:
def process_image_warp(image, points, inv=False):
    h, w = image.shape[0], image.shape[1]
    pts1 = np.float32(points)
    pts2 = np.float32([[0,0],[w,0],[0,h],[w,h]])
    if inv:
        matrix = cv2.getPerspectiveTransform(pts2, pts1)
    else:
        matrix = cv2.getPerspectiveTransform(pts1,pts2)
    img_w = cv2.warpPerspective(image, matrix, (w,h))
    return img_w

In [12]:
def process_image_histogram(image):
    histogram = np.sum(image, axis=0)
    mid_w = int(histogram.shape[0] / 2)

    leftx_max = np.argmax(histogram[:mid_w])
    rightx_max = np.argmax(histogram[mid_w:]) + mid_w

    return leftx_max, rightx_max

In [13]:
def process_sliding_window(image, leftx_pts, rightx_pts, nwindows=9):
    h, w = image.shape
    img_out = np.ones((h,w,3), dtype=np.int32)
    
    window_height = image.shape[0] // nwindows
    
    nonzero = image.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])
    
    leftx_current = leftx_pts
    rightx_current = rightx_pts
    
    left_lane_idx = []
    right_lane_idx = []
    
    margin = 60
    
    for window in range(nwindows):
        win_y_low = image.shape[0] - (window + 1) * window_height
        win_y_high = image.shape[0] - window * window_height
        win_xleft_low = leftx_current - margin
        win_xleft_high = leftx_current + margin
        win_xright_low = rightx_current - margin
        win_xright_high = rightx_current + margin
        
        cv2.rectangle(img_out, (win_xleft_low, win_y_low), (win_xleft_high, win_y_high), (0, 255, 0), 2)
        cv2.rectangle(img_out, (win_xright_low, win_y_low), (win_xright_high, win_y_high), (0, 255, 0), 2)
        
        good_left_idx = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & (nonzerox >= win_xleft_low) & (nonzerox < win_xleft_high))
        good_right_idx = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & (nonzerox >= win_xright_low) & (nonzerox < win_xright_high))
        
        good_left_idx = good_left_idx.nonzero()[0]
        good_right_idx = good_right_idx.nonzero()[0]    
        
        left_lane_idx.append(good_left_idx)
        right_lane_idx.append(good_right_idx)
        
        if len(good_left_idx) > 50:
            leftx_current = int(np.mean(nonzerox[good_left_idx]))
        if len(good_right_idx) > 50:
            rightx_current = int(np.mean(nonzerox[good_right_idx]))

    left_lane_idx = np.concatenate(left_lane_idx)
    right_lane_idx = np.concatenate(right_lane_idx)
    
    leftx = nonzerox[left_lane_idx]
    lefty = nonzeroy[left_lane_idx]
    
    rightx = nonzerox[right_lane_idx]
    righty = nonzeroy[right_lane_idx]

    left_fit = np.polyfit(lefty, leftx, 2)
    right_fit = np.polyfit(righty, rightx, 2)
    
    return left_fit, right_fit

In [14]:
alpha = 0
beta = 2
delta = 45
gamma = 130

pts = [(alpha, h), (gamma * beta, (h / 2) + delta), (w - gamma * beta, (h / 2) + delta), (w - alpha, h)]

g1, b1, e1, k1 = process_image(image1, polygon=pts)
g2, b2, e2, k2 = process_image(image2, polygon=pts)

In [15]:
widthTop = 80
heightTop = 200
widthBottom = 20
heightBottom = 220

pts_warp = np.float32([(widthTop, heightTop), (w-widthTop, heightTop), (widthBottom , heightBottom ), (w-widthBottom, heightBottom)])

w1 = process_image_warp(k1, points=pts_warp)
w2 = process_image_warp(k2, points=pts_warp)

In [27]:
warp = w1
leftx, rightx = process_image_histogram(warp)

In [28]:
left_line_lane, right_line_lane = process_sliding_window(warp, leftx, rightx, nwindows=9)


print(left_line_lane, right_line_lane)

[-8.16173033e-05  1.36252956e-01  1.46412313e+02] [2.56458196e-04 6.82123470e-02 4.82548261e+02]


In [18]:
initial_values = [0,0,0,0]

def calculate_roi(a):
    alpha = cv2.getTrackbarPos("alpha", "Trackbars")
    beta = cv2.getTrackbarPos("beta", "Trackbars")
    delta = cv2.getTrackbarPos("delta", "Trackbars")
    gamma = cv2.getTrackbarPos("gamma", "Trackbars")

    pts = [(alpha, h), (gamma * beta, (h / 2) + delta), (w - gamma * beta, (h / 2) + delta), (w - alpha, h)]

    g1, b1, e1, k1 = process_image(image1, polygon=pts)
    g2, b2, e2, k2 = process_image(image2, polygon=pts)

    widthTop = cv2.getTrackbarPos("Width Top", "Trackbars")
    heightTop = cv2.getTrackbarPos("Height Top", "Trackbars")
    widthBottom = cv2.getTrackbarPos("Width Bottom", "Trackbars")
    heightBottom = cv2.getTrackbarPos("Height Bottom", "Trackbars")

    pts_warp = [(widthTop, heightTop), (w-widthTop, heightTop), (widthBottom , heightBottom), (w-widthBottom, heightBottom)]
    
    w1 = process_image_warp(k1, points=pts_warp)
    w2 = process_image_warp(k2, points=pts_warp)
    
    cv2.imshow("process_image", util.stackImages(0.7, ([g1, k1, w1], [g2, k2, w2])))

In [19]:
cv2.namedWindow("Trackbars")
cv2.resizeWindow("Trackbars", 450, 350)
cv2.createTrackbar("alpha", "Trackbars", initial_values[0], w, calculate_roi)
cv2.createTrackbar("beta", "Trackbars", initial_values[1], 10, calculate_roi)
cv2.createTrackbar("delta", "Trackbars", initial_values[2], h, calculate_roi)
cv2.createTrackbar("gamma", "Trackbars", initial_values[3], w, calculate_roi)
cv2.createTrackbar("Width Top", "Trackbars", initial_values[0],w, calculate_roi)
cv2.createTrackbar("Height Top", "Trackbars", initial_values[1], h, calculate_roi)
cv2.createTrackbar("Width Bottom", "Trackbars", initial_values[2],w, calculate_roi)
cv2.createTrackbar("Height Bottom", "Trackbars", initial_values[3], h, calculate_roi)

cv2.namedWindow("process_image")
cv2.imshow("process_image", util.stackImages(0.7, ([g1, k1, w1], [g2, k2, w2])))
cv2.waitKey(0)
cv2.destroyAllWindows()